# Bento ML

In [1]:
import bentoml
from datetime import datetime as dt
from src.config import ROOT_DIR


## Saving Models in BentoML

In [2]:
from joblib import load
import os


model_path =os.path.join(f'{ROOT_DIR}/models/', 'random_forest_model.joblib')
model = load(model_path)

In [3]:
from pydantic_models.inference import ModelMetaData

model_name = 'rf_model'
metadata = ModelMetaData(
    model_name=model_name,
    version="1.0.0",
    trained_date=dt.now(),
)

/home/ubuntu/projects/dota2pred/dota2pred/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in ModelMetaData has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [4]:

saved_model = bentoml.sklearn.save_model(
    name=model_name,
    model=model, 
    signatures={'predict':{'batchable':True}},
    metadata=metadata.model_dump()
)

print(f"Model saved: {saved_model}")

# Model is saved at /home/user/bentoml/models/ OR ~/bentoml/models/

Model saved: Model(tag="rf_model:36ltvpa6vgqcyrlk")


In [5]:
model = bentoml.sklearn.get('rf_model:latest')
model.load_model()

RandomForestClassifier()

### Other Key arguments:

Key Arguments for save_model
The save_model function has several useful arguments:

- name: The name to identify your model (required)
- model: The actual model object to save (required)
- signatures: Specifications for model methods that will be exposed (e.g., predict, transform)
    - Batching is particularly important for performance optimization in production, as it allows BentoML to group multiple requests together for more efficient processing.
- labels: Key-value pairs for organizing models (e.g., "team": "ml-team", "project": "fraud-detection")
- custom_objects: Additional Python objects to save with the model (tokenizers, preprocessors, etc.)
- external_modules: Additional Python modules to be saved with the model
- metadata: Custom metadata to associate with the model

In [6]:
# Viewing list of saved Models

model = bentoml.models.get('rf_model:latest')
model

Model(tag="rf_model:36ltvpa6vgqcyrlk", path="/home/ubuntu/bentoml/models/rf_model/36ltvpa6vgqcyrlk")

In [7]:
model.info.metadata

{'model_name': 'rf_model',
 'version': '1.0.0',
 'trained_date': '2025-04-21T12:12:17.040981',
 'previous_version': '',
 'changes': {'changes': [],
  'performance_metrics': {'accuracy': 0.0,
   'f1_score': 0.0,
   'precision': 0.0,
   'recall': 0.0},
  'training_data': ''}}

# Running a BentoML service

In [ ]:
import bentoml 
from bentoml.models import BentoModel
import numpy as np
from typing import Dict, Any

# configurations
my_image = bentoml.images.Image(python_version='3.10', distro='debian') \
    .run('echo "Installing system packages..."') \
    .system_packages('curl') \
    .requirements_file('requirements.txt') \
    .run('echo "Image Built Successfully...!"')

@bentoml.service(name='match_prediction', image=my_image)
class MatchPredictionService:
    
    rf_model = BentoModel('rf_model:latest')
    
    def __init__(self):
        self.model = bentoml.sklearn.load_model(self.rf_model)
        
    @bentoml.api
    def predict(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        
        features = np.array(input_data["features"])
        
        prediction = self.model.predict(features)
        
        return {"prediction": prediction.tolist()}

# Serving the Model

for local deployment, run `bento serve` in the directory which contains the service.py file

## Building a Bento Container with Python SDK

In [ ]:
try:
    bento = bentoml.build(
    service='match_prediction',
    )
    print(f'{bento} has been successfully created')
except Exception as e:
    print(f'failed to build bento, error: {e}')
    
try:
    bentoml.container.build(
        bento_tag=str(bento.tag),
        image_tag=('match_prediction:latest',)
    )
    print(f'Container for {bento.tag} built successfully')
except Exception as e:
    print(f'failed to create container for bento {bento}: {e}')